## Unit Test Using Functions

In [15]:
import pandas as pd
import os
from dotenv import load_dotenv

load_dotenv()
AZURE_OPENAI_ENDPOINT    = os.getenv("AZURE_OPENAI_ENDPOINT")
AZURE_OPENAI_API_KEY     = os.getenv("AZURE_OPENAI_API_KEY")
AZURE_OPENAI_DEPLOYMENT  = os.getenv("AZURE_OPENAI_DEPLOYMENT")

In [2]:
# Pull user-property data using the Genie Data Query Agent

questions = [
    """Retrieve customer that saved at least 2 properties in the past 7 days. 
    Find the last 2 properties they saved and last 2 properties they viewed."""

    , """Retrieve the last 4 saved or viewed properties in the past 14 days 
    for each customer. Deduplicate between saved and viewed list giving saved 
    higher priority. If there are overlaps, keep properties in saved and 
    continue searching for viewed to fill the list. Label each property with 
    their action type."""

    , """Retrieve customers that registered for any auction events in the past 7 days. 
    Get top 2 properties for those events."""

    , """Get 4 least engaged (save or view) properties for the upcoming auction events."""

]

from genie_audience_property_query import build_query_prompt
query_prompt = build_query_prompt(questions[1])
print(query_prompt)


    Retrieve the last 4 saved or viewed properties in the past 14 days 
    for each customer. Deduplicate between saved and viewed list giving saved 
    higher priority. If there are overlaps, keep properties in saved and 
    continue searching for viewed to fill the list. Label each property with 
    their action type.
    Always include these property detail columns: ListingPrice, StreetNumber,
    StreetName, City, State, County, TransactionType, SQFT. Drop properties that
    do not have all the details.
    Include other columns relevant to the question.
    Drop customers that do not meet the property count threshold.
    


In [3]:
# This is calling the custom python function to query the Genie Space
# For the front end app, if you already have a function to call the Genie Space
# And process the data, use that, don't have to use this.
# Just pass the data output to the next part for content generation
from genie_audience_property_query import genie_audience_property_query

result = genie_audience_property_query(query_prompt)

query       = result["query"]
df          = result["result_table"]
result_json = result["result_json"]
conversation_id = result["conversation_id"]
attachment = result["attachment"]

Status: FILTERING_CONTEXT (0s elapsed)
Status: ASKING_AI (5s elapsed)
Status: ASKING_AI (10s elapsed)
Status: ASKING_AI (15s elapsed)
Status: COMPLETED (20s elapsed)


In [7]:
# Writing instruction options for the highlight writer
instructions = [
    "follow this format: $ListingPrice | City, State | TransactionType | Beds | Baths | SQFT"
    , "with captivating messaging encouraging customer engagements. Do not exceed 50 words"
    , "with captivating messaging encouraging customers to participate in the auction. Do not exceed 50 words"
]

The next few cells prepares the data for the highlight writer, can calls the writer to generate the property highlights.

If the front end app gets the writing instructions and the audience-property data from a different source, remember to pass the data in the appropriate functions.

In [25]:
# Prepare the data (deduplicate the property list)
# This function can only take dataframe as the input

from property_content_generator import dedupe_properties
exclude_cols = ['xome_user_id','event_datetime','action_type'] # in most cases these are the only columns that need to be excluded
dedupe_keys = ['listing_id'] # sometimes this column may be called ListingID
unique_properties = dedupe_properties(df, exclude_cols, dedupe_keys)
unique_properties.shape

(1306, 9)

In [31]:
# Build the prompt for the highlight writer
# This test uses the "simple" version 
# Needs input: the deduplicated property list, the writing instruction
from property_content_generator import format_property_for_prompt
from property_content_generator import build_prompt_simple
from property_content_generator import write_property_highlight
from openai import AzureOpenAI

client = AzureOpenAI(
    azure_endpoint=AZURE_OPENAI_ENDPOINT,
    api_key=AZURE_OPENAI_API_KEY,
    api_version="2025-01-01-preview",
)
model = AZURE_OPENAI_DEPLOYMENT

instruction = instructions[1]

results = []

# unique_properties has 1300 rows. 
# Just to demo the result, run the first 20 rows only to save time

for _, row in unique_properties.head(20).iterrows():
    prompt = build_prompt_simple(row, instruction)    
    highlight = write_property_highlight(client, model, prompt)
    results.append({**row.to_dict(), "highlight": highlight})

df_highlights = pd.DataFrame(results)
df_highlights.head()

,listing_id,ListingPrice,StreetNumber,StreetName,City,State,County,TransactionType,SQFT,highlight
0,418114618,425000,4370,8TH,Los Angeles,CA,LOS ANGELES,REO,1445,Discover this charming Los Angeles home at 437...
1,416118529,155000,2929,YORKSHIRE,Phoenix,AZ,MARICOPA,REO,1215,Discover your next investment opportunity! Thi...
2,417022331,325000,279,MARKWOOD,Chandler,AZ,MARICOPA,REO,2007,"Discover your dream home at 279 Markwood, Chan..."
3,418297233,510000,20140,PIENZA,Northridge,CA,LOS ANGELES,REO,1563,Discover your dream home in Northridge! This s...
4,418213869,120000,5144,WASHINGTON,Lake Wales,FL,POLK,NEWLY_FORECLOSED,1605,Discover this newly foreclosed gem in Lake Wal...


In [ ]:
# join property highlight result back to the full audience-property table.
cols = list(dict.fromkeys([c for c in exclude_cols + ["listing_id"] if c in df.columns]))
df_result = df[cols].merge(df_highlights, on="listing_id", how="left")
df_result.head()

,xome_user_id,event_datetime,action_type,listing_id,ListingPrice,StreetNumber,StreetName,City,State,County,TransactionType,SQFT,highlight
0,1407444,2026-03-25 15:36:16.298827 UTC,save,418114618,425000,4370,8TH,Los Angeles,CA,LOS ANGELES,REO,1445,Discover this charming Los Angeles home at 437...
1,1407444,2026-03-25 15:34:59.472153 UTC,save,416118529,155000,2929,YORKSHIRE,Phoenix,AZ,MARICOPA,REO,1215,Discover your next investment opportunity! Thi...
2,1407444,2026-03-28 15:40:06.383144 UTC,view,417022331,325000,279,MARKWOOD,Chandler,AZ,MARICOPA,REO,2007,"Discover your dream home at 279 Markwood, Chan..."
3,1407444,2026-03-28 15:34:53.395786 UTC,view,418297233,510000,20140,PIENZA,Northridge,CA,LOS ANGELES,REO,1563,Discover your dream home in Northridge! This s...
4,1462212,2026-03-24 15:37:15.737717 UTC,save,418213869,120000,5144,WASHINGTON,Lake Wales,FL,POLK,NEWLY_FORECLOSED,1605,Discover this newly foreclosed gem in Lake Wal...
